# Analiz raporu: neyi denedik, ne çıktı

**Soru:** XGBoost tabanlı varyans tahmini + standardize residual kantili (filtered historical simulation), basit baseline'lardan (GARCH(1,1), EWMA, historical VaR) daha iyi bir VaR üretiyor mu?

Bu rapor altyapıyı değil **tickerleri, ağırlıkları, feature'ları ve sonuçları** anlatır; deneyler sırayla, her adımda bir önceki adımın bıraktığı soruyla ilerler. Her tablo `report_tables.py` içindeki bir fonksiyondan gelir (kayıtlı run'lardan ve kayıtlı deneylerden okunur, hiçbir şey yeniden eğitilmez), böylece aynı fonksiyonlar sonra bir "Analysis report" sayfasında çağrılabilir. Anlatım metinleri bu notebook'ta; düzenleyebilirsin.

### Tabloları nasıl okumalı
| Sütun | Anlamı |
|---|---|
| mean tick loss | VaR'ın skoru: aşım kaybı + gereksiz sermaye kaybı. **Düşük iyi.** |
| vs baseline % | Aynı portföyün baseline'ına (GARCH) göre mean tick loss farkı. **Negatif = baseline'dan iyi.** |
| exceedances / expected | Gerçekleşen aşım sayısı / beklenen aşım sayısı. |
| tests | Kupiec (aşım oranı), Christoffersen bağımsızlık (aşımlar kümeleniyor mu) ve koşullu kapsama testlerinin **üçü de** geçilmişse `pass`. |
| DM p / Holm p | Günlük tick loss farkının Diebold-Mariano (HAC) testi; Holm, aynı tablodaki karşılaştırma sayısı için düzeltilmiş hali. |
| in MCS | %90 Model Confidence Set içinde mi: en iyiden ayırt edilemeyen run'lar. |

**VaR nasıl okunuyor:** VaR = −(μ̂ + σ̂ · q); σ̂ modelin portföy varyansı tahmininin karekökü, q önceki 252 standardize residual'ın (en az 30) (1 − güven) kantili. İlk ~252 tahmin gününde kantil, ilk pencerenin eğitim günlerindeki residual'lardan gelir. Aksi söylenmedikçe VaR %95, test günleri 2019-03-04 → 2026-06-30 (1842 gün).

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import report_tables as rt

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
show = rt.styled

## 1. Evren: tickerler, pencereler, ağırlıklar

- **4 hisse:** AAPL, JPM, XOM, KO, eşit ağırlık (0,25). İlk dört adım bu portföyle.
- **13 hisse:** AAPL, MSFT, NVDA, AVGO, GOOGL, META, NFLX, AMZN, HD, MCD, PG, KO, WMT.
- **23 hisse:** 13 hisse + XOM, CVX, JPM, GS, BAC, V, JNJ, UNH, LLY, MRK. Eşit ağırlık (1/23) ve beş **rastgele sabit ağırlıklı** portföy (rw1–rw5): Dirichlet çekimi, tohumlu (tekrarlanabilir), dönem boyunca sabit. Alfa küçüldükçe portföy konsantre oluyor: etkin ticker sayısı (1 / Σwᵢ²) 11,6'dan 1,3'e iniyor.
- **İki walk-forward ailesi:** 2010'dan başlayanlar (basit feature'lar, 4 hisse; eğitim 252 / 504 / 756, tahmin 252 ve 21 gün) ve **2016-03-01'den başlayan** (gerçekleşen varyans, RV, verisi 5 dakikalık barlardan 2016'dan itibaren olduğu için): eğitim 756 gün, tahmin 21 gün, 88 pencere. RV hedefli tüm deneyler ikincisinde.

In [2]:
show(rt.universes())

,tickers,n tickers,train bars,forecast bars,first forecast day,last forecast day,windows,forecast days,price data,runs
0,"AAPL, JPM, XOM, KO",4.00,252.00,252.00,2011-02-16,2026-06-30,16.00,3864.00,5af40f,8.00
1,"AAPL, JPM, XOM, KO",4.00,504.00,252.00,2012-02-16,2026-06-30,15.00,3612.00,5af40f,8.00
2,"AAPL, JPM, XOM, KO",4.00,756.00,21.00,2013-02-20,2026-06-30,160.00,3360.00,5af40f,8.00
3,"AAPL, JPM, XOM, KO",4.00,756.00,21.00,2019-03-04,2026-06-30,88.00,1842.00,5af40f,21.00
4,"AAPL, JPM, XOM, KO",4.00,756.00,21.00,2019-03-04,2026-06-30,88.00,1842.00,b24343,13.00
5,"AAPL, JPM, XOM, KO",4.00,756.00,252.00,2013-02-20,2026-06-30,14.00,3360.00,5af40f,8.00
6,13 tickers,13.00,756.00,21.00,2019-03-04,2026-06-30,88.00,1842.00,9ab792,8.00
7,23 tickers,23.00,756.00,21.00,2019-03-04,2026-06-30,88.00,1842.00,6f385e,48.00


In [3]:
show(rt.portfolios(), digits=2)

,n tickers,effective tickers,largest weight,smallest weight,Dirichlet alpha,seed
portfolio,,,,,,
"4 tickers, equal",4.00,4.00,AAPL 0.25,0.25,,
"23 tickers, equal (1/23)",23.00,23.00,AAPL 0.04,0.04,,
rw1,23.00,11.60,META 0.17,0.00,1.00,11.00
rw2,23.00,10.10,BAC 0.23,0.00,1.00,22.00
rw3,23.00,6.80,MCD 0.24,0.00,0.30,33.00
rw4,23.00,4.50,LLY 0.40,0.00,0.30,44.00
rw5,23.00,1.30,MCD 0.86,0.00,0.10,55.00


## 2. Modeller, feature'lar, hedefler

**Baseline'lar (hep portföy getirisi üzerinde, "wide"):**
- `garch_wide`: GARCH(1,1), normal dağılım, sıfır ortalama.
- `ewma`: RiskMetrics EWMA (λ = 0,94), hiçbir model yok.
- `historical`: son 252 portföy getirisinin kantili (varyans tahmini yok; bu yüzden QLIKE hesaplanmaz).

**XGBoost:** `long` biçimi tek bir havuzlanmış model: her ticker'ın satırı tek modelde, hedef ticker varyansı; portföy varyansı `w'Σw` ile kurulur (Σ = D·R·D: D modelin ticker volatilitesi, yani varyans tahmininin karekökü, R EWMA kovaryansının korelasyonu). `wide` biçimi portföyün kendi getirisi üzerinde.

**Taşınabilir feature'lar:** `MarketEwmaVol` portföyün kendi EWMA volatilitesi, yani ağırlığa bağlı. Bir modeli başka bir portföye (başka ağırlık, başka ticker) uygulamak için bu blok çıkarılıyor ("portable"). 23 hisseyle eğitilen modeller hep taşınabilir.

In [4]:
show(rt.model_catalog(), digits=0)

,model,family,format,target,scale,portable features,ticker counts
0,garch_wide,garch,wide,-,,,"4, 13, 23"
1,ewma,ewma,wide,-,,,"4, 13, 23"
2,historical,historical,wide,-,,,"4, 13, 23"
3,xgb_wide,xgboost,wide,SquaredReturn,10000,False,4
4,xgb_long,xgboost,long,SquaredReturn,10000,False,4
5,xgb_long,xgboost,long,SquaredReturn,10000,True,"4, 13, 23"
6,xgb_long_n300,xgboost,long,SquaredReturn,10000,False,4
7,xgb_long_ratio,xgboost,long,RatioToBase,1,False,4
8,xgb_long_ratio_ticker,xgboost,long,RatioToBase,1,False,4
9,xgb_long_drv_har,xgboost,long,RelativeRvChange,1,False,4


In [5]:
rt.feature_matrix()

,Returns,AbsReturn,RollingStats,EwmaVol,MarketEwmaVol,TickerOneHot,RvHar,RvLags,Vix,Sp500Return,YieldSpread10y3m,YieldSpread10y5y,YieldSpread30y10y
model,,,,,,,,,,,,,
xgb_long,x,x,x,x,x,,,,,,,,
xgb_long (portable),x,x,x,x,,,,,,,,,
xgb_long_drv_har,x,x,x,x,x,,x,,,,,,
xgb_long_drv_har (portable),x,x,x,x,,,x,,,,,,
xgb_long_drv_har_ticker,x,x,x,x,x,x,x,,,,,,
xgb_long_drv_macro,x,x,x,x,x,,x,,x,x,x,x,x
xgb_long_drv_macro (portable),x,x,x,x,,,x,,x,x,x,x,x
xgb_long_drv_macro_lags,x,x,x,x,x,,x,x,x,x,x,x,x
xgb_long_drv_macro_ticker,x,x,x,x,x,x,x,,x,x,x,x,x


In [6]:
rt.glossary_features()

,what it is
feature block,
Returns,the day's return of the ticker
AbsReturn,its absolute value
RollingStats,"volatility, mean absolute return and skew over the last 5, 10 and 21 days"
EwmaVol,the RiskMetrics EWMA volatility of the ticker (span 21)
MarketEwmaVol,"the EWMA volatility of the PORTFOLIO, the same value on every ticker row (depends on the weights, so a model that ha..."
TickerOneHot,one 0/1 column per ticker: a level per ticker for the pooled model
RvHar,"the ticker's daily realized variance (the sum of squared 5-minute returns), its 5-day and 22-day means (HAR)"
RvLags,"the realized variance of the ticker 1, 2 and 3 days earlier"
Vix,the VIX level and its 5-day percent change


In [7]:
rt.glossary_targets()

,what it is
target,
SquaredReturn,the next day's squared return times a scale: the plain variance target
RatioToBase,the next day's squared return divided by the EWMA variance: the model learns a multiplicative correction of the EWMA
RealizedVariance,"the next day's realized variance (sum of squared 5-minute returns), times a scale: a much less noisy target than r²"
RelativeRvChange,"the relative change of the realized variance from today to the next day, RV_next / RV - 1"


## 3. Adım 1: 4 hisse, basit feature'lar, model araması (2010'dan itibaren)

**Soru:** Basit feature'larla (getiri, |getiri|, 5/10/21 günlük istatistikler, EWMA vol, portföy EWMA vol) bir XGBoost modeli GARCH ya da EWMA'yı geçebilir mi?
**Denenen:** `xgb_wide`, `xgb_long`, `xgb_long_n300` (300 ağaç), `xgb_long_ratio` ve `xgb_long_ratio_ticker` (hedef: ertesi gün r² / EWMA varyansı, ticker one-hot'lu ve onsuz); eğitim 252 / 504 / 756, tahmin 252 ve 21 gün. Feature'lar hepsinde aynı basit küme; körlemesine model araması.

In [8]:
runs = rt.walk_forward_runs(2013, 756, 252)
show(rt.window_table(runs, "garch_wide", 0.95, rt.FULL_SAMPLE_PERIODS))

full sample  ... after the crash (2020-07-01 on)
                      mean tick loss  ...                   vs baseline %
garch_wide                  0.001170  ...                        0.000000
xgb_long_ratio_ticker       0.001199  ...                        0.917371
xgb_long_ratio              0.001205  ...                        1.277714
ewma                        0.001212  ...                        1.592511
xgb_long                    0.001259  ...                        3.266463
xgb_wide                    0.001269  ...                        7.665446
historical                  0.001302  ...                        4.039261
xgb_long_n300               0.001326  ...                        6.835800

[8 rows x 6 columns]

In [9]:
show(rt.window_table(runs, "garch_wide", 0.99, rt.FULL_SAMPLE_PERIODS))

full sample  ... after the crash (2020-07-01 on)
                      mean tick loss  ...                   vs baseline %
garch_wide                  0.000352  ...                        0.000000
xgb_long_ratio_ticker       0.000364  ...                        2.240401
xgb_long_ratio              0.000369  ...                        3.491625
ewma                        0.000373  ...                        2.124531
xgb_long                    0.000417  ...                       11.304976
xgb_wide                    0.000430  ...                       24.167425
historical                  0.000451  ...                       19.292074
xgb_long_n300               0.256367  ...                   112675.585781

[8 rows x 6 columns]

**Okuma (eğitim 756, tahmin 252):**
- Tüm örneklemde `garch_wide` hem %95'te hem %99'da en düşük tick loss'a sahip. En iyi XGBoost, ratio hedefli `xgb_long_ratio_ticker`: %95'te +%2,5, %99'da +%3,4. Kare getiri hedefli modeller (`xgb_long`, `xgb_wide`) %95'te +%8, %99'da +%18–22.
- **Covid çöküşünde modeller ayrışıyor:** %95'te ratio modeller GARCH'tan +%21–24 kötü, kare getiri hedefli modeller +%34–95. Çöküşten sonra fark neredeyse kayboluyor (ratio modeller +%0,9–1,3, EWMA +%1,6).
- `xgb_long_n300` %99'da bozuluyor (mean tick loss ≈ 0,256): ertesi günün varyans tahmini sıfıra yakın çıktığında standardize residual patlıyor, bu da filtered historical simulation'ın küçük bir varyans tahminine ne kadar kırılgan olduğunu gösteriyor.
- Diğer kurulumlarda (eğitim 252 / 504) sıralama aynı; eğitim penceresi uzadıkça ratio modeller hafifçe yaklaşıyor. Her 21 günde yeniden eğitmek de (tablo aşağıda) işe yaramıyor.

In [10]:
show(rt.window_table(rt.walk_forward_runs(2013, 756, 21), "garch_wide", 0.95, rt.FULL_SAMPLE_PERIODS))

full sample  ... after the crash (2020-07-01 on)
                      mean tick loss  ...                   vs baseline %
garch_wide                  0.001176  ...                        0.000000
ewma                        0.001212  ...                        1.822320
xgb_long_ratio              0.001237  ...                        1.441945
xgb_long_ratio_ticker       0.001238  ...                        1.564832
historical                  0.001302  ...                        4.274604
xgb_long                    0.001309  ...                        4.189146
xgb_wide                    0.001311  ...                        8.637738
xgb_long_n300               0.001372  ...                        6.378376

[8 rows x 6 columns]

**Kayıtlı deney: Covid sonrası (2020-07'den itibaren), eğitim 756 / tahmin 252.** Çöküşten sonra hiçbir XGBoost GARCH'tan anlamlı biçimde iyi değil; yalnızca `xgb_long_n300` ve `xgb_wide` GARCH'tan düzeltmeden sonra da anlamlı biçimde kötü. `historical` testleri geçemiyor.

In [11]:
show(rt.experiment_table("Post-Covid"))

,mean tick loss,vs baseline %,exceedances,expected,tests,DM p,Holm p,in MCS
garch_wide,0.00118,0.0,79.00,75.30,pass,,,yes
xgb_long_ratio_ticker,0.00119,0.9,83.00,75.30,pass,0.377,0.690,yes
xgb_long_ratio,0.00119,1.3,84.00,75.30,pass,0.234,0.690,yes
ewma,0.00119,1.6,87.00,75.30,pass,0.172,0.690,yes
xgb_long,0.00121,3.3,76.00,75.30,pass,0.111,0.557,yes
historical,0.00122,4.0,58.00,75.30,fail,0.187,0.690,yes
xgb_long_n300,0.00126,6.8,75.00,75.30,pass,0.003,0.019,yes
xgb_wide,0.00127,7.7,70.00,75.30,pass,0.003,0.019,yes


**Sonuç (Adım 1):** Basit feature'larla körlemesine model araması baseline'ları geçmedi. Modeller çöküşte ayrışıyor; kare getiri hedefi en kötüsü. Bu yüzden hedef tanımına bakıldı.

## 4. Adım 2: gerçekleşen varyans (RV) hedefi ve HAR feature'ları (4 hisse, 2019-03 → 2026-06)

**Soru:** Gürültülü r² yerine ertesi günün gerçekleşen varyansını (5 dakikalık barların kareleri toplamı) hedef almak, ve RV'nin 5 / 22 günlük ortalamalarını (HAR) feature yapmak VaR'ı iyileştirir mi?
**Denenen:** `xgb_long_rv` (RV hedefi), `xgb_long_rv_har` (+ HAR), `xgb_long_drv_har` (hedef: RV'nin göreli değişimi RV_next / RV − 1, + HAR), karşılaştırma için `xgb_long` (kare getiri) ve `xgb_long_ratio_ticker`.

In [12]:
show(rt.experiment_table("RV target vs squared"))

,mean tick loss,vs baseline %,exceedances,expected,tests,DM p,Holm p,in MCS
xgb_long_drv_har,0.00129,-3.5,95.00,92.10,pass,0.045,0.182,yes
xgb_long_rv_har,0.00131,-1.5,98.00,92.10,pass,0.516,1.000,yes
garch_wide,0.00133,0.0,94.00,92.10,pass,,,yes
xgb_long_rv,0.00135,1.5,102.00,92.10,pass,0.539,1.000,no
xgb_long_ratio_ticker,0.00140,5.4,104.00,92.10,pass,0.010,0.057,yes
ewma,0.00141,5.5,108.00,92.10,pass,0.002,0.012,no
historical,0.00152,14.3,85.00,92.10,fail,0.077,0.232,yes
xgb_long,0.00162,21.8,104.00,92.10,pass,0.016,0.078,yes


**Okuma:** Kare getiri hedefli `xgb_long` GARCH'tan %21,8 kötü (testleri geçiyor ama Holm sonrası da anlamlı değil, p = 0,078). RV hedefleri tablonun en üstüne çıkıyor: `drv_har` −%3,5, `rv_har` −%1,5; HAR'sız `rv` ise +%1,5, yani RV hedefinin tek başına değil **HAR feature'larıyla birlikte** işe yaradığı görülüyor. `drv_har` için ham DM p = 0,046 ama Holm sonrası 0,18: anlamlılık çoklu karşılaştırmada kalmıyor. EWMA GARCH'tan düzeltmeden sonra da anlamlı biçimde kötü (+%5,5, Holm 0,012). `historical` testleri geçemiyor.

**Sonuç (Adım 2):** Hedefi r²'den RV'ye çevirmek en büyük iyileşme; yön GARCH'ın biraz altına geçiş, ama istatistiksel güç zayıf.

## 5. Adım 3: piyasa feature'ları, gecikmeli RV, ticker etiketi (4 hisse)

**Soru:** RV hedefli modele hangi eklemeler yarıyor? **Denenen** (`_macro` = VIX seviyesi + 5 günlük değişimi, S&P 500 1/5/21 günlük getirisi, üç faiz farkı; `_lags` = RV'nin 1/2/3 gün gecikmesi; `_ticker` = ticker one-hot).

In [13]:
show(rt.experiment_table("RV models: market"))

,mean tick loss,vs baseline %,exceedances,expected,tests,DM p,Holm p,in MCS
xgb_long_rv_macro,0.00128,-4.2,102.00,92.10,pass,0.001,0.018,yes
xgb_long_rv_macro_lags,0.00128,-4.0,103.00,92.10,pass,0.002,0.023,yes
xgb_long_drv_macro_ticker,0.00128,-3.8,98.00,92.10,pass,0.029,0.236,yes
xgb_long_rv_macro_ticker,0.00128,-3.7,103.00,92.10,pass,0.005,0.049,yes
xgb_long_drv_macro,0.00128,-3.6,98.00,92.10,pass,0.046,0.318,yes
xgb_long_drv_har,0.00129,-3.5,95.00,92.10,pass,0.045,0.318,yes
xgb_long_drv_har_ticker,0.00129,-3.5,96.00,92.10,pass,0.058,0.318,yes
xgb_long_drv_macro_lags,0.00129,-2.9,100.00,92.10,pass,0.107,0.322,no
xgb_long_rv_har_lags,0.00130,-2.2,97.00,92.10,pass,0.319,0.638,yes
xgb_long_rv_har,0.00131,-1.5,98.00,92.10,pass,0.516,0.638,yes


**Okuma:** En iyi `xgb_long_rv_macro` (−%4,2; Holm 0,018): bu tabloda düzeltmeden sonra da GARCH'tan anlamlı biçimde iyi görünen ilk XGBoost; `rv_macro_lags` (−%4,0) ve `rv_macro_ticker` (−%3,7) da öyle ama marj çok dar. Piyasa feature'ları RV hedefinde `rv_har`'ı (−%1,5) yaklaşık 2,7 puan iyileştiriyor; **göreli değişim hedefinde (`drv_*`) etki yok** (−%3,5 → −%3,6). Gecikmeli RV ve ticker etiketi `macro` üstüne anlamlı bir şey katmıyor (−%4,2 → −%4,0 ve −%3,7); `drv_macro_lags` Model Confidence Set'in dışına bile düşüyor.

**Sonuç (Adım 3):** `har` ve `macro` varyantları tutuldu; `lags` ve `ticker` eklemeleri bırakıldı (beş modelle devam edildi: `xgb_long`, `rv_har`, `drv_har`, `rv_macro`, `drv_macro`).

## 6. Adım 4: havuz büyüklüğü: 4 hisse mi, 23 hisse mi?

**Soru:** Modeli 4 hisseyle değil 23 hisseyle bir kez eğitip AAPL/JPM/XOM/KO portföyüne **uygulamak** (yeniden eğitmeden, `_from23`) VaR'ı iyileştirir mi?
**Not:** Bu deneyde 4 hisseyle eğitilen karşılaştırma modelleri de **taşınabilir** feature kümesiyle (portföy EWMA vol'u yok) eğitildi, böylece fark yalnızca havuz büyüklüğü.

In [14]:
show(rt.experiment_table("Pool size"))

,mean tick loss,vs baseline %,exceedances,expected,tests,DM p,Holm p,in MCS
xgb_long_drv_har_from23,0.00127,-4.4,95.00,92.10,pass,0.017,0.157,yes
xgb_long_rv_macro,0.00128,-4.3,100.00,92.10,pass,0.002,0.019,yes
xgb_long_drv_macro_from23,0.00128,-3.7,98.00,92.10,pass,0.061,0.430,yes
xgb_long_drv_har,0.00128,-3.6,95.00,92.10,pass,0.063,0.430,yes
xgb_long_rv_har_from23,0.00129,-3.5,98.00,92.10,pass,0.085,0.430,yes
xgb_long_rv_macro_from23,0.00129,-3.2,101.00,92.10,pass,0.025,0.198,yes
xgb_long_drv_macro,0.00129,-2.8,98.00,92.10,pass,0.113,0.430,yes
xgb_long_rv_har,0.00131,-1.8,94.00,92.10,pass,0.477,0.954,yes
garch_wide,0.00133,0.0,94.00,92.10,pass,,,yes
xgb_long_from23,0.00135,1.5,102.00,92.10,pass,0.628,0.954,yes


**Okuma:** Beş modelin dördünde 23 hisseyle eğitilen sürüm GARCH'a daha yakın ya da daha iyi (`drv_har` −%4,4 'e karşı −%3,6; `rv_har` −%3,5 'e karşı −%1,8; `drv_macro` −%3,7 'ye karşı −%2,8); `rv_macro` tersi (−%3,2 'ye karşı −%4,3). En büyük fark kare getiri modelinde: 4 hisseyle eğitilen `xgb_long` +%19,4 ve testleri geçemiyor, 23 hisseyle eğitilen +%1,5. İki sürümü doğrudan birbirine karşı test etmedik; yön karışık, tek net kazanç kare getiri modelinde.

**Sonuç (Adım 4):** Havuzu büyütmek tutarlı bir kazanç getirmiyor ama taşınabilir modeli (başka portföye uygulanabilir) kaybetmeden veriyor. Bundan sonra 23 hisseyle bir kez eğitilmiş modeller başka portföylere uygulandı.

## 7. Adım 5: 23 hisse, eşit ağırlık (1/23)

**Soru:** Çeşitlendirilmiş 23 hisselik portföyde 23 hisseyle eğitilmiş modeller GARCH ve EWMA'yı geçiyor mu?

In [15]:
show(rt.experiment_table("23 tickers, equal weight"))

,mean tick loss,vs baseline %,exceedances,expected,tests,DM p,Holm p,in MCS
xgb_long_drv_har,0.00121,-1.7,106.00,92.10,pass,0.256,1.000,yes
xgb_long_drv_macro,0.00122,-1.1,100.00,92.10,pass,0.459,1.000,yes
xgb_long_rv_macro,0.00123,-0.1,99.00,92.10,pass,0.966,1.000,yes
garch_wide,0.00123,0.0,98.00,92.10,pass,,,yes
xgb_long_rv_har,0.00123,0.0,103.00,92.10,pass,0.992,1.000,yes
xgb_long,0.00128,4.3,98.00,92.10,pass,0.134,0.671,no
ewma,0.00131,6.8,101.00,92.10,pass,0.002,0.014,yes
historical,0.00147,19.6,88.00,92.10,fail,0.026,0.157,yes


**Okuma:** RV hedefli modellerin avantajı küçülüyor: `drv_har` −%1,7, `drv_macro` −%1,1, `rv_macro` ve `rv_har` GARCH ile aynı (−%0,06 ve +%0,02); hiçbiri anlamlı değil (Holm 1,0). Kare getiri hedefli `xgb_long` +%4,3. EWMA +%6,8 ve GARCH'tan düzeltmeden sonra da anlamlı biçimde kötü (Holm 0,014). `historical` +%19,6 ve testleri geçemiyor.

**Sonuç (Adım 5):** Portföy çeşitlendikçe XGBoost'un GARCH'a karşı avantajı küçülüyor. Bu tek bir portföy: sonucun portföy ağırlıklarına bağlı olup olmadığına bakmak için ağırlıklar değiştirildi.

## 8. Adım 6: rastgele sabit ağırlıklı beş portföy (aynı 23 hisse)

**Soru:** Aynı 23 hisselik modeller, ağırlıkları değişen portföylerde GARCH'ı geçiyor mu? Modeller **yeniden eğitilmedi**: 23 hisselik taşınabilir modeller her portföye uygulandı (`pretrained_from`), GARCH / EWMA / historical her portföyün kendi getirisiyle yeniden hesaplandı.

Tablolar: aynı portföyün GARCH'ına göre mean tick loss farkı, % (negatif = GARCH'tan iyi); "ahead in" modelin beş rastgele portföyün kaçında GARCH'tan iyi olduğunu sayıyor (eq hariç).

In [16]:
show(rt.vs_baseline_across(0.95, "whole period"), digits=1)

,eq,rw1,rw2,rw3,rw4,rw5,mean (random),ahead in
xgb_long_drv_har,-1.7,-3.3,-2.3,-2.0,-2.7,-2.0,-2.5,5 of 5
xgb_long_rv_macro,-0.1,-2.0,-1.1,-0.2,-3.1,-2.6,-1.8,5 of 5
xgb_long_rv_har,0.0,-2.2,-1.2,-1.2,-2.4,-1.8,-1.7,5 of 5
xgb_long_drv_macro,-1.1,-2.5,-1.4,-0.9,-2.1,-1.6,-1.7,5 of 5
xgb_long,4.3,1.5,2.3,2.9,0.1,3.1,2.0,0 of 5
ewma,6.8,5.1,5.6,6.9,1.2,3.6,4.5,0 of 5
historical,19.6,17.8,17.0,18.9,6.4,10.3,14.1,0 of 5


In [17]:
show(rt.vs_baseline_across(0.99, "whole period"), digits=1)

,eq,rw1,rw2,rw3,rw4,rw5,mean (random),ahead in
xgb_long_drv_har,-6.0,-9.3,-5.0,-4.5,-3.5,-5.1,-5.5,5 of 5
xgb_long_rv_har,-6.8,-9.9,-6.3,-4.3,-6.1,-0.3,-5.4,5 of 5
xgb_long_drv_macro,-3.5,-7.8,-2.7,-2.8,-3.5,-5.7,-4.5,5 of 5
xgb_long_rv_macro,-5.0,-8.6,-4.8,-2.3,-6.3,0.7,-4.2,4 of 5
xgb_long,-2.0,-4.3,-1.9,3.5,-3.8,10.3,0.8,3 of 5
ewma,0.4,1.4,2.3,4.9,-0.7,8.2,3.2,1 of 5
historical,35.1,31.1,35.5,37.5,15.2,29.3,29.7,0 of 5


**Okuma (tüm dönem):**
- **%95'te RV hedefli dört modelin dördü de beş rastgele portföyün beşinde GARCH'tan iyi**, farklar küçük: ortalama −%1,7 ile −%2,5. EWMA ve historical hiçbirinde GARCH'ı geçmiyor; kare getiri hedefli `xgb_long` da geçemiyor (0/5).
- **%99'da** farklar büyüyor (RV modeller ortalama −%4 ile −%5,5), `xgb_long` karışık (3/5), `ewma` 1/5, `historical` hep çok kötü.

In [18]:
show(rt.holm_across(0.95), digits=3)

,eq,rw1,rw2,rw3,rw4,rw5
ewma,0.014,0.175,0.083,0.075,0.796,0.153
historical,0.157,0.175,0.257,0.491,0.796,0.524
xgb_long,0.671,0.889,1.000,1.000,0.955,0.598
xgb_long_rv_har,1.000,0.889,1.000,1.000,0.467,0.598
xgb_long_drv_har,1.000,0.175,0.380,1.000,0.342,0.598
xgb_long_rv_macro,1.000,0.889,1.000,1.000,0.533,0.505
xgb_long_drv_macro,1.000,0.463,1.000,1.000,0.796,0.598


In [19]:
show(rt.holm_across(0.99), digits=3)

,eq,rw1,rw2,rw3,rw4,rw5
ewma,1.000,0.763,1.000,0.597,1.000,0.154
historical,0.359,0.294,0.369,0.597,1.000,0.753
xgb_long,1.000,0.670,1.000,0.747,1.000,0.753
xgb_long_rv_har,0.205,0.015,0.186,0.597,0.291,1.000
xgb_long_drv_har,0.205,0.002,0.186,0.409,1.000,0.753
xgb_long_rv_macro,0.293,0.025,0.369,0.747,0.489,1.000
xgb_long_drv_macro,0.513,0.033,0.767,0.747,1.000,0.753


**Anlamlılık (Holm düzeltmeli Diebold-Mariano p, aynı portföyde 7 karşılaştırma):**
- %95'te **hiçbir XGBoost modeli GARCH'tan düzeltmeden sonra anlamlı biçimde iyi değil** (en düşük Holm p = 0,17, rw1'de `drv_har`). Anlamlı olan tek şey EWMA'nın eq portföyünde GARCH'tan kötü olması (0,014).
- %99'da yalnızca **rw1** net: `drv_har` (0,002), `rv_har` (0,015), `rv_macro` (0,025), `drv_macro` (0,033). Diğer portföylerde anlamlı değil.
- **Dikkat:** Beş portföy aynı 23 hisseden ve aynı piyasa faktöründen çıktığı için **bağımsız örnek sayılmaz**; "5 / 5" göründüğü kadar güçlü bir kanıt değil.

**Covid çöküşü ve sonrası (%95):** çöküş yalnızca 93 gün (beklenen aşım ≈ 5), bu yüzden tanımlayıcı.

In [20]:
show(rt.vs_baseline_across(0.95, "Covid crash (2020-02-19 to 2020-06-30)"), digits=1)

,eq,rw1,rw2,rw3,rw4,rw5,mean (random),ahead in
xgb_long_rv_macro,3.9,4.1,3.1,2.2,-12.5,-5.1,-1.7,2 of 5
xgb_long_drv_har,3.0,0.6,-0.4,2.5,-4.6,6.1,0.8,2 of 5
xgb_long_drv_macro,5.4,3.3,2.0,5.0,-3.9,6.3,2.5,1 of 5
xgb_long_rv_har,8.9,7.3,6.6,5.7,-2.3,3.4,4.2,1 of 5
xgb_long,18.0,15.8,15.0,17.9,10.8,28.9,17.7,0 of 5
ewma,31.2,26.6,30.5,36.7,17.3,23.3,26.9,0 of 5
historical,71.1,70.3,72.1,81.6,39.3,57.6,64.2,0 of 5


In [21]:
show(rt.vs_baseline_across(0.95, "after the crash (2020-07-01 on)"), digits=1)

,eq,rw1,rw2,rw3,rw4,rw5,mean (random),ahead in
xgb_long_drv_har,-2.2,-3.9,-2.3,-2.3,-2.6,-3.5,-2.9,5 of 5
xgb_long_rv_har,-1.4,-3.7,-2.5,-2.2,-2.7,-2.7,-2.8,5 of 5
xgb_long_drv_macro,-2.0,-3.4,-1.6,-1.5,-2.0,-3.1,-2.4,5 of 5
xgb_long_rv_macro,-0.6,-3.1,-1.8,-0.4,-1.9,-2.2,-1.9,5 of 5
xgb_long,1.7,-1.2,-0.2,-0.4,-1.7,-1.7,-1.0,5 of 5
ewma,2.5,1.3,0.9,0.7,-1.4,0.1,0.3,1 of 5
historical,10.3,9.1,7.5,6.9,2.1,1.2,5.4,0 of 5


**Okuma:** Çöküşte RV modellerinin GARCH'a karşı işareti portföye göre değişiyor (örn. `rv_macro` rw4'te −%12,5, rw1'de +%4,1); EWMA ve historical çöküşte çok kötü (+%27 ve +%64 ortalama). Çöküşten sonra ise RV modelleri beş portföyün beşinde GARCH'tan %1,9–2,9 iyi, kare getiri hedefli `xgb_long` de 5/5 önde (−%1,0).

## 9. Adım 7: konsantrasyon farkı açıklıyor mu?

**Hipotez:** 23 hisselik eşit ağırlıklı portföyde portföy varyansının çoğu çapraz terimlerden (korelasyonlar) geliyor; XGBoost yalnızca ticker varyanslarını (köşegen) tahmin ettiği için avantajı küçük. O doğruysa portföy konsantre oldukça (köşegen payı arttıkça) avantaj büyümeliydi.

In [22]:
show(rt.diagonal_share_table())

,effective tickers,largest weight,diagonal share,xgb_long_drv_har vs GARCH %
portfolio,,,,
"23 tickers, equal (1/23)",23.000000,AAPL 0.04,0.190000,-1.700000
rw1,11.600000,META 0.17,0.310000,-3.300000
rw2,10.100000,BAC 0.23,0.340000,-2.300000
rw3,6.800000,MCD 0.24,0.420000,-2.000000
rw4,4.500000,LLY 0.40,0.650000,-2.700000
rw5,1.300000,MCD 0.86,0.900000,-2.000000


**Okuma:** Köşegen payı 0,19'dan 0,90'a çıkıyor ama `drv_har`'ın GARCH'a karşı farkı düz: −%1,7, −%3,3, −%2,3, −%2,0, −%2,7, −%2,0. **Hipotez desteklenmedi;** farkın nedeni bilinmiyor.

## 10. Adım 8: günlük minimum risk portföy (23 hisse)

**Soru (VaR dışında bir uygulama):** Modelin her gün ürettiği ticker varyansları ve EWMA korelasyonu ile Σ = D·R·D kurulup, her gün `w'Σw`'yi en aza indiren (uzun pozisyon, toplam 1) ağırlıklar seçilirse portföyün gerçekleşen riski ne olur? Model: `xgb_long_drv_har` (23 hisse, eşit ağırlıklı walk-forward). Ağırlık günün tahminlerinden, getiri o günün getirisinden.

In [23]:
minimum_risk = rt.minimum_risk_tables()
show(minimum_risk["realized risk"], digits=2)

,volatility (% / year),forecast volatility (% / year),realized r² / forecast variance,worst day (%),mean of the worst 5% days (%)
minimum risk (daily weights),14.85,10.58,1.98,-8.99,-2.22
equal weight (1/23),18.96,15.66,1.47,-12.31,-2.80


In [24]:
show(minimum_risk["weights by ticker"], digits=1)

,mean weight (%),max weight (%),days with weight 0 (%)
WMT,10.7,59.6,20.1
JNJ,10.1,65.4,27.5
KO,9.9,61.3,33.4
PG,8.6,82.5,31.8
MCD,8.3,64.8,30.3
CVX,6.0,38.9,47.8
MRK,6.0,49.3,44.7
XOM,4.5,31.6,47.0
UNH,4.4,43.7,46.4
LLY,4.3,40.0,44.1


In [25]:
show(minimum_risk["summary"], digits=1)

,value
days,1842.0
mean daily turnover (% of the portfolio),59.4
tickers with a weight above 1% (mean per day),9.7
largest weight of the day (mean),0.3


**Okuma:** Gerçekleşen yıllık volatilite %14,8'e iniyor (eşit ağırlıkta %19,0), en kötü gün −%9,0 (−%12,3 yerine). Ama optimizer tahminin riski **olduğundan düşük** gördüğü yerleri seçiyor: tahmin edilen volatilite %10,6 iken gerçekleşen %14,8; gerçekleşen r² / tahmin edilen varyans 1,98 (eşit ağırlıkta 1,47; 1 = kalibre). Yani Σ'dan türeyen bir VaR bu portföyde yeterince korumasız kalırdı. Ağırlıklar çok oynak (günlük devir ortalama %59) ve modelin düşük varyans tahmini verdiği hisselere (ortalama ağırlığı en büyük beşi: WMT, JNJ, KO, PG, MCD) yığılıyor; ağırlık üst sınırı ve işlem maliyeti yok.

## 11. Akışın özeti

| Adım | Soru | Ne değişti (tickerler / ağırlık / feature / hedef) | Ana sonuç |
|---|---|---|---|
| 1 | Basit feature'larla XGBoost baseline'ları geçer mi? | 4 hisse eşit; basit feature'lar; hedef r² ya da r² / EWMA | Hayır: GARCH en iyi; Covid çöküşünde modeller ayrışıyor, kare getiri en kötüsü |
| 2 | RV hedefi + HAR yardımcı olur mu? | 4 hisse, 2016 başlangıçlı walk-forward; hedef RV ya da dRV; HAR feature'ları | Evet, en büyük iyileşme: GARCH'ın %1,5–3,5 altı, ama Holm sonrası anlamlı değil |
| 3 | Piyasa feature'ları, RV gecikmeleri, ticker etiketi? | + VIX, S&P 500, faiz farkları; + RV gecikmeleri; + ticker one-hot | Piyasa feature'ları RV hedefinde ~2,7 puan; gecikme ve etiket katkı yok |
| 4 | 4 yerine 23 hisseyle eğitmek? | 23 hisseyle bir kez eğitilmiş taşınabilir model 4 hisselik portföye uygulandı | Karışık; kare getiri modeli net düzeliyor; taşınabilirlik kazanıldı |
| 5 | 23 hisselik eşit ağırlıklı portföy | 23 hisse, 1/23 | Avantaj −%1,7'ye küçülüyor, anlamlı değil; EWMA anlamlı kötü |
| 6 | Ağırlıklar değişince? | beş rastgele sabit ağırlık (etkin ticker 11,6 → 1,3) | RV modeller GARCH'tan %1–3 iyi, 5/5; yalnızca %99'da rw1'de anlamlı |
| 7 | Konsantrasyon açıklıyor mu? | köşegen payı 0,19 → 0,90 | Hayır, avantaj düz |
| 8 | Günlük minimum risk portföy (VaR dışı) | günlük değişen ağırlık, 23 hisse | Gerçekleşen vol %14,8 ama risk olduğundan düşük tahmin ediliyor |

## 12. Sınırlar

- Tek dönem (2019-03 → 2026-06), tek evren (23 ABD büyük hissesi), beş rastgele portföy: istatistiksel güç düşük, farkların çoğu çoklu karşılaştırma düzeltmesinde anlamlılığını kaybediyor.
- Rastgele portföyler bağımsız örnek değil (aynı hisseler, aynı piyasa faktörü).
- Covid çöküşü penceresi 93 gün; %95'te beklenen aşım ≈ 5: yalnızca tanımlayıcı.
- RV hedefi seans içi varyans (kapanıştan kapanışa varyansın yaklaşık %50–85'i): VaR'da kantil bunu telafi ediyor ama QLIKE gibi seviye skorlarında RV hedefli modeller adaletsiz cezalanıyor; bu rapor VaR skoru (tick loss) ve testlerle ölçüyor.
- `historical` her yerde Kupiec testini geçemiyor ya da çok kötü; filtered historical simulation'ın küçük bir varyans tahmini karşısındaki kırılganlığı (Adım 1, `xgb_long_n300` %99) bir risk.
- Henüz yapılmadı: minimum risk portföyün VaR'ı, günlük ağırlık değişim sınırı, farklı dönem ve evrenlerde tekrar.